In [ ]:
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
import warnings
warnings.filterwarnings("ignore")

import sys
#import os
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py as h5
import sklearn
from sklearn.multioutput import MultiOutputClassifier
from sklearn import metrics
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, precision_recall_curve, accuracy_score, confusion_matrix  
from sklearn.metrics import average_precision_score
import pickle
import keras
from keras.models import load_model
import sys
sys.path.append(r'C:\Users\aoara\develop\deepbeat')
import utils
from pathlib import Path
from tensorflow.keras.optimizers import Adam
from scipy.io import loadmat
import copy
#import seaborn as sns


# Load model

In [1]:
import argparse
import json
import h5py
from pathlib import Path


h5_file_path = r'C:\develop\afib_detection\keras\deepbeat.h5'
config = None
with h5py.File(h5_file_path, 'r') as f:
    training_config = json.loads(f.attrs['training_config'])
    optimizer_config = training_config['optimizer_config']
    
print(training_config)
print(optimizer_config)

{'optimizer_config': {'class_name': 'Adam', 'config': {'lr': 3.906250185536919e-06, 'beta_1': 0.8999999761581421, 'beta_2': 0.9990000128746033, 'decay': 0.0, 'epsilon': 1e-07, 'amsgrad': False}}, 'loss': {'qa_output': 'categorical_crossentropy', 'rhythm_output': 'binary_crossentropy'}, 'metrics': ['acc'], 'sample_weight_mode': None, 'loss_weights': [0.2, 5.0]}
{'class_name': 'Adam', 'config': {'lr': 3.906250185536919e-06, 'beta_1': 0.8999999761581421, 'beta_2': 0.9990000128746033, 'decay': 0.0, 'epsilon': 1e-07, 'amsgrad': False}}


In [ ]:
def get_training_config():
    with h5.File(r"C:\Users\aoara\develop\deepbeat\deepbeat.h5", 'r') as f:
    # Check for version attributes
        if 'keras_version' in f.attrs:
            print(f"Keras version: {f.attrs['keras_version']}")
        
        if 'backend' in f.attrs:
            print(f"Backend: {f.attrs['backend']}")
        
        # Sometimes stored under model config
        if 'model_config' in f.attrs:
            import json
            config = json.loads(f.attrs['model_config'])
            if 'keras_version' in config:
                print(f"Keras version (from config): {config['keras_version']}")
    
    training_config = json.loads(f.attrs['training_config'])
    return training_config 

In [ ]:
# get training config
with h5.File(r"C:\Users\aoara\develop\deepbeat\deepbeat.h5", 'r') as f:
    # Check for version attributes
    if 'keras_version' in f.attrs:
        print(f"Keras version: {f.attrs['keras_version']}")
    
    if 'backend' in f.attrs:
        print(f"Backend: {f.attrs['backend']}")
    
    # Sometimes stored under model config
    if 'model_config' in f.attrs:
        import json
        config = json.loads(f.attrs['model_config'])
        if 'keras_version' in config:
            print(f"Keras version (from config): {config['keras_version']}")
    
    # Print all available attributes
    print("\nAll attributes in file:")
    for key in f.attrs.keys():
        print(f"  {key}: {f.attrs[key]}")
    
    training_config = json.loads(f.attrs['training_config'])

In [ ]:
orig_config

In [ ]:
orig_config = training_config ['optimizer_config']['config']
orig_config['learning_rate'] = orig_config.pop('lr') # rename lr to learning rate
orig_config.pop('decay') # there is no longer a parameter called decay; the original decay was 0

# load deepbeat model with new tensorflow package, verify performances
path_to_model =r'C:\Users\aoara\develop\deepbeat'
model_name = 'deepbeat.h5'
deepbeat = load_model( Path(path_to_model) / model_name, compile = False) 

## Verify Loaded Model Performances

In [ ]:
## load original test data
data_path = Path(r'C:\Users\aoara\develop\deepbeat\data\db')
data_test = np.load(data_path / 'test.npz', allow_pickle=True)
test_x = data_test['signal']
test_qa = data_test['qa_label']
test_r = data_test['rhythm']
test_p = pd.DataFrame(data_test['parameters'])
print('test data shape: ')
print(test_x.shape)
print(test_qa.shape)
print(test_r.shape)
print(test_p.shape)
test_p.rename(index=str, columns={0:'timestamp', 
                                  1:'stream', 
                                  2:'ID'}, inplace=True)
## QA results

predictions_qa, predictions_r = deepbeat.predict(test_x)
predictions_QA = np.argmax(predictions_qa, axis=1)

#print(classification_report(np.argmax(test_qa, axis=1), predictions_QA))


excellent_qa_indx = np.where(predictions_QA==2)[0]
x_test_excellent = test_x[excellent_qa_indx,:]
p_test_excellent = test_p.iloc[excellent_qa_indx,:]
rhythm_test_excellent = test_r[excellent_qa_indx,:]
quality_assessment_test_excellent = test_qa[excellent_qa_indx,:]


# weighted macro-average across all indivduals

test_metrics_1 = utils.collecting_individual_metrics(deepbeat, x_test_excellent, p_test_excellent, rhythm_test_excellent, out_message=False)
test_metrics = pd.DataFrame.from_dict(test_metrics_1).T.rename(columns={0:'TPR', 1:'TNR', 2:'FPR', 3:'FNR', 4:"total_samples"})


for m in ['TPR', 'TNR', 'FPR', 'FNR']:
    metric_wmu = np.average(test_metrics[m][~test_metrics[m].isna()], weights=test_metrics['total_samples'][~test_metrics[m].isna()])
    print('%s: %0.2f' % (m, metric_wmu))
    
# PPV, NPV and F1

episode_m = utils.episode_metrics(deepbeat, x_test_excellent, p_test_excellent, rhythm_test_excellent, out_message=False)
episode_metrics = pd.DataFrame(episode_m).T
episode_metrics.rename(columns={0:'TPR', 1:'TNR', 2:'PPV', 3:'NPV', 4:"FPR", 5:'FNR', 6:'F1', 7:'total_samples'}, inplace=True)
for m in ['PPV', 'NPV', 'F1']:
    print('%s: %0.2f' % (m, episode_metrics[m]))

- benchmarks paper in PPG realms; see how it is constructed
-- see how they did it
-- publish this

- VSM watch version, algorithm versions

## Overview
### Step 1: Prepare data
1. deepbeat original + cleaned deepbeat data ---> took too long to update
- 1.1 for each updated subject, replace every updated labels --> took >3hr to find data matches and replace labels
- 1.2 for each updated subject, remove their old data, only use their new data


In [ ]:
import sys
from train_new_model import *
orig_data_path = r'C:\Users\aoara\datasets\db'
relabled_path = r'C:\Users\aoara\develop\deepbeat\data\labeled_data'
train_data = load_original_data(orig_data_path, 'train.npz')
relabeled_combined, relabeled_db, relabeled_vsm = load_relabeled_data(relabled_path)
# merged old data with relabeled data
#db_train_copy = copy.deepcopy(orig_train)
# db_train_update = replace_updated_subjects_db(db_train_copy, relabeled_db)
# db_VSM_train = attach_VSM(db_train_update, relabeled_vsm)

In [ ]:
from tqdm import tqdm

def chunk_and_replace_relabeled_signals(db_train, relabeled_db, chunk_size=100):
    """
    Memory-efficient version that processes relabeled segments in chunks
    """
    unique_subjects = np.unique(relabeled_db['ID'])
    
    for sub_id in tqdm(unique_subjects, desc="Processing subjects"):
        
        mask_relabeled = (relabeled_db['ID'] == sub_id)
        mask_origin = (db_train['ID'] == sub_id)
        
        # Get original and relabeled data for each subject
        orig_data = db_train['data'][mask_origin, :]
        orig_rhy = db_train['rhythm'][mask_origin, :].copy()
        orig_qa = db_train['qa_label'][mask_origin, :].copy()
        relabeled_data = relabeled_db['data'][mask_relabeled, :]
        relabeled_rhy = relabeled_db['rhythm'][mask_relabeled, :]
        relabeled_qa = relabeled_db['qa_label'][mask_relabeled, :]
        
        n_relabeled = relabeled_data.shape[0]
        n_original = orig_data.shape[0]
        
        # Process in chunks to avoid memory issues
        for chunk_start in range(0, n_relabeled, chunk_size):
            chunk_end = min(chunk_start + chunk_size, n_relabeled) 
            
            # Process only a chunk of relabeled segments at a time
            chunk_data = relabeled_data[chunk_start:chunk_end, :]
            
            # Compare this chunk against all original segments
            # Shape: (chunk_size, n_original, n_features) -> (chunk_size, n_original)
            differences = np.sum(
                orig_data[np.newaxis, :, :] - chunk_data[:, np.newaxis, :], 
                axis=2
            )
            
            # Find matches
            matches = (differences == 0)
            match_counts = np.sum(matches, axis=1)
            
            # Process based on match counts
            single_match = (match_counts == 1)
            no_match = (match_counts == 0)
            multiple_match = (match_counts > 1)
            
            # Warnings
            if np.any(no_match):
                no_match_indices = np.where(no_match)[0] + chunk_start
                for seg_i in no_match_indices:
                    tqdm.write(f"Warning: No match found for subject {sub_id}, segment {seg_i}")
            
            if np.any(multiple_match):
                multiple_match_indices = np.where(multiple_match)[0] + chunk_start
                for seg_i in multiple_match_indices:
                    tqdm.write(f"Warning: Multiple matches found for subject {sub_id}, segment {seg_i}")
            
            # Update labels where we have exactly one match
            for local_idx in np.where(single_match)[0]:
                seg_i = chunk_start + local_idx
                orig_idx = np.where(matches[local_idx])[0][0]
                orig_rhy[orig_idx, :] = relabeled_rhy[seg_i, :]
                orig_qa[orig_idx, :] = relabeled_qa[seg_i, :]
        
        # Assign back to db_train
        db_train['rhythm'][mask_origin, :] = orig_rhy
        db_train['qa_label'][mask_origin, :] = orig_qa
    
    return db_train

In [ ]:
db_train_replaced = chunk_and_replace_relabeled_signals(train_data , relabeled_db, chunk_size= 30)

##### 1.2 for updated subjects, keep their relabeled data only

In [ ]:
def replace_updated_subjects_db(db_train, relabeled_db):
    
    subjects_to_replace = np.unique(relabeled_db['ID'])
    mask_keep = ~np.isin(db_train['ID'], subjects_to_replace)
    
    db_train['data'] = db_train['data'][mask_keep]
    db_train['rhythm'] = db_train['rhythm'][mask_keep]
    db_train['qa_label'] = db_train['qa_label'][mask_keep]
    db_train['ID'] = db_train['ID'][mask_keep]
    
    db_train['data'] = np.concatenate([db_train['data'], relabeled_db['data']], axis=0)
    db_train['rhythm'] = np.concatenate([db_train['rhythm'], relabeled_db['rhythm']], axis=0)
    db_train['qa_label'] = np.concatenate([db_train['qa_label'], relabeled_db['qa_label']], axis=0)
    db_train['ID'] = np.concatenate([db_train['ID'], relabeled_db['ID']], axis=0)
     
    return db_train

def attach_VSM (db_data, relabeled_vsm):
    db_data['data'] = np.concatenate([db_data['data'], relabeled_vsm['data']], axis=0)
    db_data['rhythm'] = np.concatenate([db_data['rhythm'], relabeled_vsm['rhythm']], axis=0)
    db_data['qa_label'] = np.concatenate([db_data['qa_label'], relabeled_vsm['qa_label']], axis=0)
    db_data['ID'] = np.concatenate([db_data['ID'], relabeled_vsm['ID']], axis=0)
    return db_data
    


In [ ]:
def shuffle_data(db_train):
    """

    Args:
        db_train (dict): keys - 'data', 'qa_label', 'rhythm', 'ID'
    """
    data_train, label_train_r, label_train_q = db_train['data'], db_train['rhythm'], db_train['qa_label']
    # random shuffle
    idx = np.random.permutation(range(len(label_train_r)))  # shuffled indices
    # shuffle together
    data_train, label_train_r, label_train_q = data_train[idx, :], label_train_r[idx], label_train_q[idx]
    
    return data_train, label_train_r, label_train_q

In [ ]:
new_db = keras.models.clone_model(deepbeat)
new_db.compile(
    optimizer= Adam(
        **orig_config
    ),
    loss={
        'qa_output': 'categorical_crossentropy',
        'rhythm_output': 'binary_crossentropy' ### Samiya used BinaryFocalLoss(gamma=2)
    },
    loss_weights={
        'qa_output': 0.2,      
        'rhythm_output': 5.0   
    },
    metrics={'rhythm_output': 'accuracy', 'qa_output': 'accuracy'}
)

#  How Samiya compiled the compact DB :
# model.compile(optimizer=tf.keras.optimizers.Adam(),
#             loss={'rhythm_output': BinaryFocalLoss(gamma=2), 'qa_output': 'categorical_crossentropy'},
#             loss_weights={'rhythm_output': 1, 'qa_output': 1},
#             metrics={'rhythm_output': 'accuracy', 'qa_output': 'accuracy'})
